# Bihar's Economy: A Sectoral and Comparative Analysis

This notebook analyses Bihar's economic structure using official data from the **Directorate of Economics and Statistics (DES), Government of Bihar**, collected during an internship at DES Patna.

**Data source:** DES Bihar: Gross State Value Added (GSVA) and GSDP series, base year 2011-12 (MoSPI methodology)  
**Coverage:** 2011-12 to 2024-25 (2023-24 provisional, 2024-25 quick estimates)

### Key Questions
1. How is Bihar's sectoral structure shifting over time?
2. How does Bihar's structure compare to India's national economy?
3. How large and persistent is the per capita income gap?
4. How volatile is Bihar's growth relative to India's?

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# --- Plot style ---
plt.rcParams.update({
    'font.family': 'DejaVu Sans',
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.grid': True,
    'grid.alpha': 0.3,
    'figure.dpi': 120
})

# Colour palette
BIHAR_PRIMARY   = '#2E86AB'
BIHAR_SECONDARY = '#A23B72'
BIHAR_TERTIARY  = '#F18F01'
INDIA_COLOR     = '#4CAF50'
GAP_COLOR       = '#E8E8E8'

print('Libraries loaded.')

## 1. Load and Inspect Data

In [ ]:
# Load all datasets
bihar_shares  = pd.read_csv('data/bihar_sectoral_shares_constant.csv')
india_shares  = pd.read_csv('data/india_sectoral_shares.csv')
per_capita    = pd.read_csv('data/per_capita_income.csv')
growth_rates  = pd.read_csv('data/gsdp_growth_rates.csv')

# Quick sanity check: sectoral shares should sum to ~100
bihar_shares['total'] = bihar_shares[['primary_pct','secondary_pct','tertiary_pct']].sum(axis=1)
print('Bihar sectoral shares (should sum ~100):')
print(bihar_shares[['year','primary_pct','secondary_pct','tertiary_pct','total']].to_string(index=False))

# Drop helper column
bihar_shares.drop(columns='total', inplace=True)

## 2. Chart 1: Bihar's Structural Shift (Sectoral Shares Over Time)

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))

years = bihar_shares['year']
x = range(len(years))

ax.plot(x, bihar_shares['primary_pct'],   marker='o', color=BIHAR_PRIMARY,   linewidth=2.2, label='Primary (Agriculture & Mining)')
ax.plot(x, bihar_shares['secondary_pct'], marker='s', color=BIHAR_SECONDARY,  linewidth=2.2, label='Secondary (Industry & Construction)')
ax.plot(x, bihar_shares['tertiary_pct'],  marker='^', color=BIHAR_TERTIARY,   linewidth=2.2, label='Tertiary (Services)')

# Annotate start and end values for primary and secondary (the story)
ax.annotate(f"{bihar_shares['primary_pct'].iloc[0]}%",
            xy=(0, bihar_shares['primary_pct'].iloc[0]), xytext=(-0.3, 1.5),
            textcoords='offset points', fontsize=9, color=BIHAR_PRIMARY)
ax.annotate(f"{bihar_shares['primary_pct'].iloc[-1]}%",
            xy=(len(years)-1, bihar_shares['primary_pct'].iloc[-1]), xytext=(3, -10),
            textcoords='offset points', fontsize=9, color=BIHAR_PRIMARY)
ax.annotate(f"{bihar_shares['secondary_pct'].iloc[0]}%",
            xy=(0, bihar_shares['secondary_pct'].iloc[0]), xytext=(3, -12),
            textcoords='offset points', fontsize=9, color=BIHAR_SECONDARY)
ax.annotate(f"{bihar_shares['secondary_pct'].iloc[-1]}%",
            xy=(len(years)-1, bihar_shares['secondary_pct'].iloc[-1]), xytext=(3, 3),
            textcoords='offset points', fontsize=9, color=BIHAR_SECONDARY)

ax.set_xticks(x)
ax.set_xticklabels(years, rotation=45, ha='right', fontsize=9)
ax.set_ylabel('Share of GSDP (%)', fontsize=11)
ax.set_title("Bihar's Sectoral Composition of GSDP (Constant 2011-12 Prices)",
             fontsize=13, fontweight='bold', pad=15)
ax.legend(loc='center right', fontsize=10)
ax.set_ylim(10, 65)

# Add source note
fig.text(0.01, -0.02, 'Source: Directorate of Economics and Statistics, Government of Bihar | Base Year: 2011-12',
         fontsize=8, color='grey')

plt.tight_layout()
plt.savefig('charts/01_bihar_sectoral_shift.png', bbox_inches='tight', dpi=150)
plt.show()
print('Chart saved.')

**Reading:** Bihar's primary sector share has fallen from **25.2% → 17.7%** while the secondary sector has risen from **18.4% → 25.9%**, a clear structural shift toward industry and construction. Services remain dominant but relatively stable around 53–56%.

## 3. Chart 2 - Bihar vs India: Sectoral Structure Compared

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5), sharey=False)

sectors = [
    ('primary_pct',   'Primary Sector',   BIHAR_PRIMARY),
    ('secondary_pct', 'Secondary Sector',  BIHAR_SECONDARY),
    ('tertiary_pct',  'Tertiary Sector',   BIHAR_TERTIARY),
]

years = bihar_shares['year']
x = range(len(years))

for ax, (col, title, color) in zip(axes, sectors):
    ax.plot(x, bihar_shares[col],  color=color,       linewidth=2.2, label='Bihar',  marker='o', markersize=4)
    ax.plot(x, india_shares[col],  color='#555555',   linewidth=2.2, label='India',  marker='s', markersize=4, linestyle='--')
    ax.set_title(title, fontsize=11, fontweight='bold')
    ax.set_xticks(x)
    ax.set_xticklabels(years, rotation=90, fontsize=7)
    ax.set_ylabel('Share of GDP/GSDP (%)', fontsize=9)
    ax.legend(fontsize=9)

fig.suptitle('Sectoral Composition: Bihar vs India (Constant 2011-12 Prices)',
             fontsize=13, fontweight='bold', y=1.02)
fig.text(0.01, -0.05, 'Source: DES Bihar & MoSPI | Base Year: 2011-12',
         fontsize=8, color='grey')

plt.tight_layout()
plt.savefig('charts/02_bihar_vs_india_sectors.png', bbox_inches='tight', dpi=150)
plt.show()
print('Chart saved.')

**Reading:** Bihar and India are converging on secondary sector share. Bihar's primary sector dependence, though falling, tracks close to India's — reflecting India's own atypical rise in agricultural share post-2018. The tertiary sector gap is notable: India's services are deeper (IT, finance, formal trade) while Bihar's are largely public administration and informal trade.

## 4. Chart 3: The Per Capita Income Gap

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))

years = per_capita['year']
x = range(len(years))

india_vals = per_capita['india_per_capita_gdp_current']
bihar_vals = per_capita['bihar_per_capita_gsdp_current']

# Shaded gap
ax.fill_between(x, bihar_vals, india_vals, alpha=0.12, color='#888888', label='Income gap')

ax.plot(x, india_vals, color=INDIA_COLOR,     linewidth=2.5, marker='s', markersize=5, label='India (Per Capita GDP)')
ax.plot(x, bihar_vals, color=BIHAR_PRIMARY,   linewidth=2.5, marker='o', markersize=5, label='Bihar (Per Capita GSDP)')

# Ratio annotation: Bihar as % of India each year
for i, (b, ind) in enumerate(zip(bihar_vals, india_vals)):
    ratio = round((b / ind) * 100, 1)
    if i % 3 == 0:  # annotate every 3rd year to avoid clutter
        ax.annotate(f'{ratio}%', xy=(i, b), xytext=(0, -18),
                    textcoords='offset points', fontsize=8,
                    color=BIHAR_PRIMARY, ha='center')

ax.set_xticks(x)
ax.set_xticklabels(years, rotation=45, ha='right', fontsize=9)
ax.set_ylabel('Per Capita Income (₹, Current Prices)', fontsize=11)
ax.set_title('Per Capita Income Gap: Bihar vs India (Current Prices)',
             fontsize=13, fontweight='bold', pad=15)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'₹{int(x):,}'))
ax.legend(fontsize=10)

# Callout note
ax.annotate('Bihar\'s per capita income\nstuck at ~30–35% of India\'s\nthroughout this period',
            xy=(7, bihar_vals.iloc[7]),
            xytext=(8.5, 90000),
            arrowprops=dict(arrowstyle='->', color='gray'),
            fontsize=9, color='gray')

fig.text(0.01, -0.02, 'Source: DES Bihar & MoSPI | Note: % labels show Bihar as share of India per capita income',
         fontsize=8, color='grey')

plt.tight_layout()
plt.savefig('charts/03_per_capita_gap.png', bbox_inches='tight', dpi=150)
plt.show()
print('Chart saved.')

**Reading:** Bihar's per capita income has grown from ₹23,525 to ₹76,490, but so has India's (₹63,462 to ₹1,59,280). The ratio has barely moved: Bihar remains at roughly **30–35% of the national average** throughout the period. High GSDP growth is not translating into per capita convergence, pointing to structural issues in labour productivity and formal employment generation.

## 5. Chart 4: Growth Rate Volatility - Bihar vs India

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))

years = growth_rates['year']
x = np.arange(len(years))
width = 0.35

bars_bihar = ax.bar(x - width/2, growth_rates['bihar_gsdp_growth'],
                    width, label='Bihar GSDP Growth', color=BIHAR_PRIMARY, alpha=0.85)
bars_india = ax.bar(x + width/2, growth_rates['india_gdp_growth'],
                    width, label='India GDP Growth',  color=INDIA_COLOR,   alpha=0.85)

# Reference line at 0
ax.axhline(0, color='black', linewidth=0.8)

# Highlight COVID year
covid_idx = list(years).index('2020-21')
ax.axvspan(covid_idx - 0.5, covid_idx + 0.5, alpha=0.08, color='red', label='COVID-19 year')

ax.set_xticks(x)
ax.set_xticklabels(years, rotation=45, ha='right', fontsize=9)
ax.set_ylabel('Growth Rate (%, Constant 2011-12 Prices)', fontsize=11)
ax.set_title('Annual GSDP/GDP Growth Rate: Bihar vs India',
             fontsize=13, fontweight='bold', pad=15)
ax.legend(fontsize=10)

# Std deviation annotation to quantify volatility
bihar_std = growth_rates['bihar_gsdp_growth'].std()
india_std = growth_rates['india_gdp_growth'].std()
ax.text(0.02, 0.95,
        f'Bihar std dev: {bihar_std:.1f}pp  |  India std dev: {india_std:.1f}pp',
        transform=ax.transAxes, fontsize=9, color='gray',
        verticalalignment='top')

fig.text(0.01, -0.02, 'Source: DES Bihar & MoSPI | Base Year: 2011-12 | pp = percentage points',
         fontsize=8, color='grey')

plt.tight_layout()
plt.savefig('charts/04_growth_volatility.png', bbox_inches='tight', dpi=150)
plt.show()
print('Chart saved.')

**Reading:** Bihar shows significantly higher growth volatility than India (higher standard deviation). While Bihar outpaces India in several years, the swings are sharper; both upward and downward. This is consistent with Bihar's catch-up economy status: a small base amplifies percentage changes, and heavy dependence on agriculture and construction makes the economy sensitive to monsoon patterns and public expenditure cycles.

## 6. Summary of Key Findings

In [ ]:
print('=' * 65)
print('KEY FINDINGS: Bihar Economy Analysis')
print('=' * 65)

# 1. Structural shift
primary_change   = bihar_shares['primary_pct'].iloc[-1]   - bihar_shares['primary_pct'].iloc[0]
secondary_change = bihar_shares['secondary_pct'].iloc[-1] - bihar_shares['secondary_pct'].iloc[0]
print(f'\n1. STRUCTURAL SHIFT (2011-12 to 2024-25)')
print(f'   Primary sector:   {bihar_shares["primary_pct"].iloc[0]}% → {bihar_shares["primary_pct"].iloc[-1]}%  (Δ {primary_change:+.1f}pp)')
print(f'   Secondary sector: {bihar_shares["secondary_pct"].iloc[0]}% → {bihar_shares["secondary_pct"].iloc[-1]}%  (Δ {secondary_change:+.1f}pp)')

# 2. Per capita gap
latest = per_capita.iloc[-1]
earliest = per_capita.iloc[0]
ratio_start = (earliest['bihar_per_capita_gsdp_current'] / earliest['india_per_capita_gdp_current']) * 100
ratio_end   = (latest['bihar_per_capita_gsdp_current']   / latest['india_per_capita_gdp_current'])   * 100
print(f'\n2. PER CAPITA INCOME GAP')
print(f'   Bihar as % of India (2011-12): {ratio_start:.1f}%')
print(f'   Bihar as % of India (2024-25): {ratio_end:.1f}%')
print(f'   Gap has barely closed despite strong GSDP growth.')

# 3. Growth volatility
print(f'\n3. GROWTH VOLATILITY')
print(f'   Bihar GSDP growth std dev: {growth_rates["bihar_gsdp_growth"].std():.2f} percentage points')
print(f'   India GDP growth std dev:  {growth_rates["india_gdp_growth"].std():.2f} percentage points')
print(f'   Bihar is ~{growth_rates["bihar_gsdp_growth"].std()/growth_rates["india_gdp_growth"].std():.1f}x more volatile than India.')

# 4. CAGR of secondary sector share
n = len(bihar_shares) - 1
sec_cagr = ((bihar_shares['secondary_pct'].iloc[-1] / bihar_shares['secondary_pct'].iloc[0]) ** (1/n) - 1) * 100
print(f'\n4. SECONDARY SECTOR CAGR (share)')
print(f'   CAGR of secondary sector share: {sec_cagr:.2f}% per year')

print('\n' + '=' * 65)